# Skeletonized Morphology Visualization Notebook

Copyright (c) 2025 Open Brain Institute

+ Author(s): 
    - Michael W. Reimann < michael.reimann@openbraininstitute.org >
    - Marwan Abdellah < marwan.abdellah@openbraininstitute.org >

Last modified: 11.2025

## Imports and setting up platform authentication

We begin by importing required packages.

In [ ]:
import os
import obi_auth
import obi_notebook.get_entities
from obi_notebook import get_projects, get_entities, get_environment

from entitysdk import Client
from entitysdk.models import CellMorphology, Subject, EMCellMesh

from morph_spines import load_morphology_with_spines
import morph_spines_visualizer

import pandas as pd
from IPython.display import display

## Authentication and project selection
We authenticate with the OBI platform.
### Project selection
Select from the dropdown menu the project that the output (a morphology with extracted spines) should be registered to. It will be available only in that project context. 

The widget lists all projects you have access to.

In [ ]:
environment = get_environment.get_environment()
token = obi_auth.get_token(environment=environment, auth_mode="daf")
project_context = get_projects.get_projects(token)

## Set up clients

With the information provided above we assemble a client that can interact with the entity database.

In [ ]:
client = Client(environment=environment, token_manager=token, project_context=project_context)

## Display skeletonized morphologies table

Display a list of all the existing morphologies.

In [ ]:
microns_subject = client.search_entity(entity_type=Subject, query={"name": "IARPA MICrONS mouse"}).one()
cell_ids = []
cell_ids = obi_notebook.get_entities.get_entities("cell-morphology", token=token, result=cell_ids,
                                                  env=environment, project_context=project_context, page_size=50)

## Morphology selection

Select the morphology for visualization.

In [ ]:
root = os.path.join(os.environ["HOME"], "skeletonization_download")
os.makedirs(root, exist_ok=True)

cell_entity = client.get_entity(entity_id=cell_ids[0], entity_type=CellMorphology, 
                                project_context=project_context)
assets = [asset for asset in cell_entity.assets if asset.label == "morphology_with_spines"]
if len(assets) == 0:
    raise(RuntimeError("The selected morphology does not have spines. Please select a 'morphology-with-spines!'"))

path_dl_morph = client.download_file(entity_id=cell_entity.id, entity_type=CellMorphology, asset_id=assets[0].id, output_path=root)

## Mesh download

The EM reconstructed mesh of the selected morphology will be downloaded and combined in the visualization for visual validation.

In [ ]:
# Get the pt_root_id of the mesh 
pt_root_id = str(cell_entity.assets[0].path).rsplit('.', 1)[0].replace("microns-mesh-", "").replace("_with_spines", "").replace("-morphology", "") 

# Search for the mesh 
microns_mesh = list(client.search_entity(
    entity_type=EMCellMesh, query={"dense_reconstruction_cell_id": pt_root_id}))[0]

# Extract the full path
path_dl_mesh = f"{root}/{os.path.basename(microns_mesh.assets[0].full_path)}"

# Download the mesh file if it does not exist
if not os.path.exists(path_dl_mesh):
    path_dl_mesh = client.download_file(
         entity_id=microns_mesh.id,
         entity_type=EMCellMesh,
         asset_id=microns_mesh.assets[0].id,
         output_path=root)

## Visualize the spiny morphology 
This visualization plots the skeleton of the resulting morphology combined with the EM mesh. Users can then select any section with spines, and the spine meshes will pop-up in the scene. 

In [ ]:
# Visualize the data 
morph_spines_visualizer.visualize_morphology_with_point_cloud(
    morphology_path=path_dl_morph, 
    mesh_path=path_dl_mesh
)